## Inverse Kinemaics - Jacobian based methods 
- Test IK methods based on Jacobian
    - IK can be solved by multiplying Inverse Matrix of Jacobian
    - Several methods to solve inverse jacobian without singularity

Create UR environment

In [1]:
import mujoco
import mujoco_viewer # new viewer
import numpy as np
import time

In [2]:
model_path = "../ur5e_mjcf/scene.xml"

# declare model & data
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

Get Mujoco Jacobian

In [3]:
def get_jac_body_name(body_name=None):

    # initialize positional & rotational jacobian
    Jacobian_p = np.zeros((3,model.nu))
    Jacobian_r = np.zeros((3,model.nu))

    # get jacobian of end-effector
    mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
    return Jacobian_p, Jacobian_r

### Function: calculate inverse jacobian

In [4]:
def get_invjac_name(body_name, method='svd', upper_bound=0.001, damping=1.0):

    jacobian_p, jacobian_r = get_jac_body_name(body_name)

    if method=='svd':
        # get inverse jacobian with Singular Value Decomposition
        U, Sigma, V = np.linalg.svd(jacobian_p, compute_uv=True)

        Sigma_clipped_rev = np.minimum(1 / Sigma, upper_bound)
        # Sigma_clipped_rev = np.zeros_like(Sigma)
        # for i, value in enumerate(Sigma):
        #     if 1/Sigma[i] > upper_bound:
        #         Sigma_clipped_rev[i] = 0
        #     else:
        #         Sigma_clipped_rev[i] = 1/Sigma[i]

        # inverse matrix for position jacobian
        S_rev_matrix = np.zeros((model.nu,3)) # positional dimension = 3, dof = model.nu
        for i, value in enumerate(Sigma_clipped_rev):
            S_rev_matrix[i,i] = value
        J_inverse = V @ S_rev_matrix @ U.T

    if method=='DLS':
        # apply damped least squares
        pass
    
    return J_inverse

### MAIN loop: calculate error & update with forward

In [ ]:
""" MAIN LOOP """

# create python viewer object
viewer = mujoco_viewer.MujocoViewer(model, data)

# initialize robot
init_qpos = [0.0, -0.8, -2.5, -2.2, 0.0, 0.0]

mujoco.mj_resetData(model, data)
data.qpos = init_qpos
mujoco.mj_forward(model, data) # first forward to get jacobian with no error

# goal position
goal_pos = [0.4, 0.4, 0.2]
body_name = "wrist_3_link"


while True:
    if viewer.is_alive:

        # get inverse jacobian & unit error vector
        J_inverse = get_invjac_name(body_name=body_name, method='svd')
        error = goal_pos - data.body(body_name).xpos.copy()
        error /= np.linalg.norm(error)
        print(f"current error: {error}")

        dq = J_inverse @ error
        data.qpos -= dq
        # print(f"qpos before update: {qpos_before} \n qpos after update: {data.qpos}")

        mujoco.mj_forward(model, data)

        print(f"current body pos: {data.body(body_name).xpos}")

        # terminalize
        if np.linalg.norm(goal_pos - data.body(body_name).xpos) < 0.02:
            print("IK done.")
            break

        viewer.render()

    else:
        break

# close
viewer.close()

current error: [ 0.67885912  0.71387725 -0.17184168]
current body pos: [-0.13387072 -0.16174204  0.33513767]
current error: [ 0.67865481  0.71408474 -0.17178658]
current body pos: [-0.13374115 -0.16193815  0.33510214]
current error: [ 0.67845039  0.71429224 -0.17173137]
current body pos: [-0.13361127 -0.16213408  0.33506648]
current error: [ 0.67824587  0.71449974 -0.17167606]
current body pos: [-0.1334811  -0.16232982  0.33503067]
current error: [ 0.67804123  0.71470724 -0.17162063]
current body pos: [-0.13360457 -0.16234024  0.33530077]
current error: [ 0.67807961  0.71459555 -0.17193386]
current body pos: [-0.13347428 -0.16253616  0.33526487]
current error: [ 0.67787483  0.71480316 -0.17187829]
current body pos: [-0.13359786 -0.16254676  0.33553497]
current error: [ 0.67791314  0.71469148 -0.17219135]
current body pos: [-0.13346746 -0.16274287  0.33549898]
current error: [ 0.67770822  0.71489921 -0.17213566]
current body pos: [-0.13333677 -0.16293879  0.33546285]
current error: [ 0.